#Data Analysis & Machine Learning Project

This project demonstrates a complete data science workflow, including data preprocessing, visualization, and machine learning modeling.

The goal of this project is to analyze the dataset, extract insights, and build a predictive model using Python.

#Key Steps
* Data Cleaning and Preprocessing

* Exploratory Data Analysis (EDA)

* Feature Engineering

* Model Training

* Model Evaluation

# **1. Project Scenario**

"CityBike Systems" is facing a problem: In some stations, bikes run out during rush hour, while in others, they sit unused.
Your manager wants a data-driven solution. You need to build a model that predicts **how many bikes will be rented at any given hour** based on:

* Weather conditions (Temperature, Rain, Humidity).
* Time factors (Hour of day, Season, Holiday or Working Day).

# **2. Dataset**

* **Source:** UCI Machine Learning Repository / Kaggle.
* **Dataset Name:** [Seoul Bike Sharing Demand Dataset](https://www.kaggle.com/datasets/saurabhshahane/seoul-bike-sharing-demand-prediction)
* **Target Variable:** `Rented Bike Count`


# **3. Step-by-Step Requirements**

# Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# **Phase 1: Data Engineering & Preprocessing (The Heavy Lifting)**

* **Load & Rename:** Load the dataset. The column names are messy (e.g., `Temperature(°C)`). Rename them to clean identifiers (e.g., `Temperature`, `Humidity`, `Solar_Rad`).
* **Date Manipulation (Crucial):**
1. The dataset has a `Date` column (String format).
2. Convert it to `datetime` objects.
3. **Feature Extraction:** Create new columns from the Date:
 `Month` `Day` `Weekday` (Mon, Tue, Wed...)
4. **Categorical Encoding:**
* Convert `Seasons`, `Holiday`, and `Functioning Day` into numerical values using One-Hot Encoding (`pd.get_dummies`) or Label Encoding.

In [ ]:
df = pd.read_csv('SeoulBikeData.csv', encoding='unicode_escape')

In [ ]:
print(df.info())

In [ ]:
column_rename_mapping = {
    'Date': 'Date',
    'Rented Bike Count': 'Rented_Bike_Count',
    'Hour': 'Hour',
    'Temperature(°C)': 'Temperature',
    'Humidity(%)': 'Humidity',
    'Wind speed (m/s)': 'Wind_Speed',
    'Visibility (10m)': 'Visibility',
    'Dew point temperature(°C)': 'Dew_Point_Temperature',
    'Solar Radiation (MJ/m2)': 'Solar_Radiation',
    'Rainfall(mm)': 'Rainfall',
    'Snowfall (cm)': 'Snowfall',
    'Seasons': 'Seasons',
    'Holiday': 'Holiday',
    'Functioning Day': 'Functioning_Day'
}
df.rename(columns=column_rename_mapping, inplace=True)
print("DataFrame columns renamed successfully:")
print(df.columns)

In [ ]:
print(df.info())

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.day_name()

df = pd.get_dummies(df, columns=['Seasons', 'Holiday', 'Functioning_Day'], drop_first=False)

print("Date manipulation and categorical encoding complete.")
print(df.head())
print(df.info())

# **Phase 2: Exploratory Data Analysis (EDA) - "The Business Report"**

* *Answering Management Questions with Visuals:*
1. **Hourly Trend:** Plot a Line Chart of `Rented Bike Count` vs. `Hour`. *Identify the two peak "Rush Hours".*
2. **Weather Impact:** Use a Scatter Plot or Box Plot to show the relation between `Temperature` and `Bike Count`. *Do people rent bikes when it's freezing?*
3. **Seasonal Pattern:** Which season has the highest demand? (Bar Chart).
4. **Correlation Matrix:** Use a Heatmap to see which columns correlate most with the `Bike Count`.

In [ ]:
# 1. Hourly Trend: Plot a Line Chart of Rented Bike Count vs. Hour.
hourly_bike_demand = df.groupby('Hour')['Rented_Bike_Count'].mean().reset_index()

plt.figure(figsize=(14, 7))
sns.lineplot(x='Hour', y='Rented_Bike_Count', data=hourly_bike_demand, marker='o')
plt.title('Average Rented Bike Count by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Average Rented Bike Count')
plt.xticks(range(0, 24))
plt.grid(True)

# Identify the two peak "Rush Hours"
peak_hours = hourly_bike_demand.nlargest(2, 'Rented_Bike_Count')
for i, row in peak_hours.iterrows():
    plt.axvline(x=row['Hour'], color='r', linestyle='--', lw=1, label=f'Peak Hour: {int(row['Hour'])}h')
    plt.text(row['Hour'] + 0.2, row['Rented_Bike_Count'], f'{int(row['Hour'])}h', color='red')

plt.legend(['Average Bike Count', 'Peak Hours'])
plt.show()

print(f"The two peak rush hours are: {peak_hours['Hour'].astype(int).tolist()}")

In [ ]:
# 2. Weather Impact: Scatter Plot of Temperature and Bike Count
plt.figure(figsize=(12, 7))
sns.scatterplot(x='Temperature', y='Rented_Bike_Count', data=df, alpha=0.6)
plt.title('Rented Bike Count vs. Temperature')
plt.xlabel('Temperature (°C)')
plt.ylabel('Rented Bike Count')
plt.grid(True)
plt.show()

# Analyze rentals in freezing temperatures
freezing_temps_df = df[df['Temperature'] <= 0]
if not freezing_temps_df.empty:
    avg_rentals_freezing = freezing_temps_df['Rented_Bike_Count'].mean()
    print(f"\nAnalysis for Freezing Temperatures (<= 0°C):")
    print(f"Average Rented Bike Count in freezing temperatures: {avg_rentals_freezing:.2f}")
    if avg_rentals_freezing > 0:
        print("Yes, people still rent bikes even when it's freezing, though the count might be lower.")
    else:
        print("It appears people do not rent bikes when it's freezing.")
else:
    print("No data available for freezing temperatures (<= 0°C).")

In [ ]:
# 3. Seasonal Pattern: Which season has the highest demand? (Bar Chart).

# Calculate average bike count for each season
seasonal_demand = pd.Series({
    'Autumn': df[df['Seasons_Autumn'] == True]['Rented_Bike_Count'].mean(),
    'Spring': df[df['Seasons_Spring'] == True]['Rented_Bike_Count'].mean(),
    'Summer': df[df['Seasons_Summer'] == True]['Rented_Bike_Count'].mean(),
    'Winter': df[df['Seasons_Winter'] == True]['Rented_Bike_Count'].mean()
})

# Convert to DataFrame for easier plotting with seaborn
seasonal_demand_df = seasonal_demand.reset_index()
seasonal_demand_df.columns = ['Season', 'Average_Rented_Bike_Count']

# Plotting
plt.figure(figsize=(10, 6))
sns.barplot(x='Season', y='Average_Rented_Bike_Count', data=seasonal_demand_df, palette='viridis')
plt.title('Average Rented Bike Count by Season')
plt.xlabel('Season')
plt.ylabel('Average Rented Bike Count')
plt.grid(axis='y')
plt.show()

# Identify season with highest demand
highest_demand_season = seasonal_demand_df.loc[seasonal_demand_df['Average_Rented_Bike_Count'].idxmax()]
print(f"The season with the highest demand is: {highest_demand_season['Season']} with an average of {highest_demand_season['Average_Rented_Bike_Count']:.2f} bikes.")

In [ ]:
# 4. Correlation Matrix: Use a Heatmap to see which columns correlate most with the Bike Count.

# Select only numerical columns for correlation calculation
# Exclude boolean columns for now as they are handled differently by corr() and might clutter the heatmap
numerical_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(15, 10))
sns.heatmap(numerical_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# Display correlations with Rented_Bike_Count specifically
print("\nCorrelation with Rented_Bike_Count:")
print(numerical_df.corr()['Rented_Bike_Count'].sort_values(ascending=False))

# **Phase 3: Advanced Feature Engineering**

* **Binning:** Create a new column called `Day_Type`:
* If it's a Weekend or Holiday -> 'Leisure'
* If it's a Working Day -> 'Work'
* **Drop Redundant Columns:** Drop the original `Date` column and any columns with extremely low correlation (e.g., if `Dew point temperature` is too similar to `Temperature`, drop one to avoid multicollinearity).

In [ ]:
# Create Day_Type column based on Weekend or Holiday
is_weekend = df['Weekday'].isin(['Saturday', 'Sunday'])
is_holiday = df['Holiday_Holiday'] == True

conditions = [
    (is_weekend | is_holiday),
    (~(is_weekend | is_holiday))
]
choices = ['Leisure', 'Work']

df['Day_Type'] = np.select(conditions, choices, default='Work') # Defaulting to 'Work' for safety, though conditions should cover all cases

print("New 'Day_Type' column created:")
print(df[['Date', 'Weekday', 'Holiday_Holiday', 'Day_Type']].head())
print(df['Day_Type'].value_counts())

In [ ]:
print(df.info())

In [ ]:
# One-hot encode 'Day_Type'
df = pd.get_dummies(df, columns=['Day_Type'], drop_first=False)

# Drop redundant columns
columns_to_drop = ['Date', 'Dew_Point_Temperature', 'Weekday']
df.drop(columns=columns_to_drop, inplace=True)

print("Redundant columns dropped and Day_Type encoded.")
print(df.head())
print(df.info())

# **Phase 4: Machine Learning (Model Training & Comparison)**

* **Split Data:** Split your dataset into **75% Training** and **25% Testing** sets.
1. **Model 1: Linear Regression**
* Initialize and train a `LinearRegression` model.
* **Coefficients Analysis:** Print the coefficients (`model.coef_`).
* *Question:* Which feature has the highest positive impact on demand?

In [ ]:
# Define features (X) and target (y)
X = df.drop('Rented_Bike_Count', axis=1)
y = df['Rented_Bike_Count']

# Convert boolean columns to int for scikit-learn compatibility
X = X.astype(float)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Initialize and train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("\nLinear Regression Model Coefficients:")
coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_})
coefficients['Absolute_Coefficient'] = np.abs(coefficients['Coefficient'])
print(coefficients.sort_values(by='Coefficient', ascending=False))

# Identify feature with highest positive impact
highest_positive_impact_feature = coefficients[coefficients['Coefficient'] > 0].sort_values(by='Coefficient', ascending=False).iloc[0]
print(f"\nFeature with the highest positive impact on demand: {highest_positive_impact_feature['Feature']} (Coefficient: {highest_positive_impact_feature['Coefficient']:.2f})")


2. **Model 2: Random Forest Regressor (The Challenger)**
* Initialize and train a `RandomForestRegressor` (suggested: `n_estimators=100`).
* **Feature Importance:** Plot or print the `feature_importances_` of the Random Forest model.
* *Question:* Does Random Forest agree with Linear Regression on the most important features?

In [ ]:
# Initialize and train RandomForestRegressor model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print("\nRandom Forest Regressor Feature Importances:")
feature_importances = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
print(feature_importances.sort_values(by='Importance', ascending=False))

# Compare with Linear Regression's most important feature
# Linear Regression's highest positive impact feature was 'Functioning_Day_Yes'

print("\nComparison with Linear Regression:")
# Get top 3 features from Linear Regression coefficients by absolute value for a fair comparison across positive/negative impact
top_lr_features = coefficients.sort_values(by='Absolute_Coefficient', ascending=False).head(3)['Feature'].tolist()
print(f"Top features by absolute coefficient in Linear Regression: {top_lr_features}")

top_rf_features = feature_importances.sort_values(by='Importance', ascending=False).head(3)['Feature'].tolist()
print(f"Top 3 features by importance in Random Forest: {top_rf_features}")

if 'Functioning_Day_Yes' in top_rf_features:
    print("Random Forest also identifies 'Functioning_Day_Yes' as a very important feature.")
else:
    print("Random Forest's top features differ from Linear Regression's most impactful feature ('Functioning_Day_Yes').")

# Further check if 'Hour' and 'Temperature' are important in RF, as they were also high in LR
if 'Hour' in top_rf_features and 'Temperature' in top_rf_features:
    print("Both models agree on the importance of 'Hour' and 'Temperature'.")
elif 'Hour' in top_rf_features or 'Temperature' in top_rf_features:
    print("Random Forest agrees with Linear Regression on the importance of either 'Hour' or 'Temperature', but not both in the top 3.")
else:
    print("Random Forest has different top features compared to Linear Regression.")


# **Phase 5: Evaluation & Final Submission**

1. **Performance Metrics:**
* Calculate **RMSE** (Root Mean Squared Error) and **R2 Score** for **BOTH** models.
* **Comparison:** Create a small table or print statement comparing the results.
* *Example:* "Linear Regression R2: 0.55 | Random Forest R2: 0.82"
* *Question:* Which model is better? Why do you think that is?

In [ ]:
# Make predictions on the test set
y_pred_lr = lr_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

# Calculate RMSE and R2 Score for Linear Regression
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

# Calculate RMSE and R2 Score for Random Forest
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("\n--- Model Performance Comparison ---")
print(f"Linear Regression:  RMSE = {rmse_lr:.2f} | R2 Score = {r2_lr:.2f}")
print(f"Random Forest:      RMSE = {rmse_rf:.2f} | R2 Score = {r2_rf:.2f}")

print("\n--- Which model is better? ---")
if r2_rf > r2_lr:
    print("The Random Forest Regressor is better than the Linear Regression model.")
    print("This is likely due to its ability to capture non-linear relationships and interactions between features, ")
    print("which Linear Regression cannot do as effectively. Random Forests are ensemble models, meaning they combine ")
    print("multiple decision trees to produce a more robust and accurate prediction.")
elif r2_lr > r2_rf:
    print("The Linear Regression model is better than the Random Forest Regressor.")
    print("This could indicate that the relationship between features and the target is predominantly linear, ")
    print("or that the Random Forest model might be overfitting or its hyperparameters aren't optimally tuned.")
else:
    print("Both models performed similarly.")


# **The "Real World" Test (Simulation):**

* Create a specific DataFrame for a hypothetical scenario:
* *Scenario:* It's a **Monday** in **Summer**, **8:00 AM**, **25°C**, **No Rain**.
* *(Note: Monday is a 'Work' day, Summer is 'Seasons_Summer' = 1)*


* **Task:** Use **BOTH** models to predict the demand for this specific hour.
* **Comparison:** Compare the two predicted numbers. Which one feels more realistic based on your analysis?

Run Below Cells as it to generate sample data to test with

In [ ]:
# Create a dictionary matching the Model's Input Columns (X.columns)
# We initialize with zeros
scenario_data = {col: 0 for col in X.columns}

In [ ]:
# Fill in the specific values
scenario_data['Hour'] = 8
scenario_data['Temperature'] = 25
scenario_data['Humidity'] = 60      # Assumption (not specified, using average)
scenario_data['Wind_Speed'] = 1.5   # Assumption
scenario_data['Visibility'] = 1500  # Assumption
scenario_data['Solar_Radiation'] = 0.5 # Morning sun
scenario_data['Seasons_Summer'] = 1
scenario_data['Day_Type_Work'] = 1  # Monday is Work
scenario_data['Functioning_Day_Yes'] = 1

In [ ]:
# Convert dictionary to DataFrame (1 row)
scenario_df = pd.DataFrame([scenario_data])

In [ ]:
# Predict using Linear Regression
prediction = lr_model.predict(scenario_df)
# Predict using Random Forest
prediction_rf = rf_model.predict(scenario_df)
print(f"\n--- Real World Prediction Comparison ---")
print(f"Conditions: Summer Monday, 8:00 AM, 25C")
print(f"Linear Regression Prediction: {int(prediction[0])} bikes")
print(f"Random Forest Prediction:     {int(prediction_rf[0])} bikes")